In [132]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
from langchain.retrievers import ParentDocumentRetriever
import os

In [21]:
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0)

In [22]:
loaders = [
    TextLoader("./paul_graham_essay.txt")
    #TextLoader("../../state_of_the_union.txt"),
]
docs = []
for l in loaders:
    docs.extend(l.load())

In [111]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

In [112]:
retriever_pc = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [136]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=0)
texts = text_splitter.split_documents(docs)

Created a chunk of size 508, which is longer than the specified 500
Created a chunk of size 777, which is longer than the specified 500
Created a chunk of size 557, which is longer than the specified 500
Created a chunk of size 587, which is longer than the specified 500
Created a chunk of size 622, which is longer than the specified 500
Created a chunk of size 775, which is longer than the specified 500
Created a chunk of size 604, which is longer than the specified 500
Created a chunk of size 618, which is longer than the specified 500
Created a chunk of size 520, which is longer than the specified 500
Created a chunk of size 602, which is longer than the specified 500
Created a chunk of size 1004, which is longer than the specified 500
Created a chunk of size 1203, which is longer than the specified 500
Created a chunk of size 844, which is longer than the specified 500
Created a chunk of size 910, which is longer than the specified 500
Created a chunk of size 674, which is longer t

In [137]:
embeddings = OpenAIEmbeddings()
db = Chroma.from_documents(texts, embeddings)

In [138]:
retriever = db.as_retriever(search_kwargs={"k":4})

In [139]:
#retriever.add_documents(docs)

In [172]:
#db.similarity_search('What were some specific examples of why Paul enjoyed taking art classes at Harvard while in the PhD program for computer science?')

In [173]:
#retriever.get_relevant_documents('What were some specific examples of why Paul enjoyed taking art classes at Harvard while in the PhD program for computer science?')

In [142]:
def combine_page_contents(list_of_dicts):
    # This function assumes each dictionary in the list has a key 'page content'
    combined_content = "\n".join(d.page_content for d in list_of_dicts)
    return combined_content

In [143]:
def context_generator(chat_history, question, response):
    # 
    context = llm.predict(f"""
    Update the summary of a conversation based on the new information.
    "chat_history" : {chat_history}
    "human's question": {question}
    "AI answer": {response}
    """
    )
    return context

In [163]:
def question_generator(new_question, question, answer):
    # take chat context and new question to generate a question
    response = llm.predict(f"""
    Use the given context and a follow up question posed by a human to generate a new standalone question with enough context
    for a stateless observor to understand the question and answer. 
    If there is no previous question and answer, just output the new question.
    "previous question" : {new_question}
    "previous answer" : {answer}
    "next question": {question}
    \nStandalone Question:
    """
    )
    return response

In [164]:
new_question = ''
response = ''

In [165]:
def answer_question(
    doc_retriever,
    question="what is life?",
    ):
    """
    Answer a question based on the most similar context from the dataframe texts
    """
    global new_question
    global response
    new_question = question_generator(new_question, question, response)
    print('new question:')
    print(new_question)
    document_list = doc_retriever.get_relevant_documents(new_question)
    text_context = combine_page_contents(document_list)
    #print('context:')
    #print(text_context)

    # Create a chat completion using the question and context
    response = llm.predict(f"""
    You are a helpful conversational assistant who answers questions based on the given context. If it can't be 
    answered based on the context, say \"I don't know\"\n\n",
    "Context: {text_context}\n\n---\n\nQuestion: {new_question}\nAnswer:
    """
    )

    return response

In [166]:
answer_question(retriever, "Did paul attend grad school?")

new question:
Did Paul attend grad school?


'Yes, Paul attended grad school at Harvard.'

In [167]:
answer_question(retriever, "Did he consider any other school?")

new question:
Did Paul consider any other school besides Harvard for grad school?


'Yes, Paul considered MIT and Yale for grad school.'

In [168]:
answer_question(retriever, "Why didn't he end up going to them?")

new question:
Why didn't Paul end up going to MIT or Yale for grad school?


"Paul didn't end up going to MIT or Yale for grad school because only Harvard accepted him."

In [169]:
answer_question(retriever, "Did he enjoy it there?")

new question:
Did Paul enjoy attending Harvard for grad school?


'Based on the given context, it is not explicitly mentioned whether Paul enjoyed attending Harvard for grad school.'

In [170]:
answer_question(retriever, "What classes did he take there?")

new question:
What classes did Paul take at Harvard for grad school?


'The context does not provide information about the specific classes Paul took at Harvard for grad school.'